# Game Radar - Google Colab Runner

This notebook runs the **Game Radar** market intelligence pipeline in Google Colab (Linux) to enrich all candidates without limits.

> **Important SQLite Performance Note**:
> SQLite databases should **never** be mounted directly over Google Drive's FUSE filesystem (`/content/drive/MyDrive/...`) during multi-hour scraping runs. Drive FUSE file locking causes database lock timeouts, corruption, and extreme I/O latency.
>
> This runner copies `radar.db` to `/content/radar.db` (fast local NVMe/SSD storage) before execution, runs `python -m radar run --all-candidates`, and uses a `try...finally` block plus periodic syncs to safely flush `radar.db` and reports back to Google Drive.

### Step 1: Mount Google Drive
Mount your Google Drive to load persistent database snapshots and save generated reports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Set Up Directory & Persistent Storage Paths

In [ ]:
import os
from pathlib import Path

# Google Drive persistent storage directory
DRIVE_RADAR_DIR = Path("/content/drive/MyDrive/GameRadar")
DRIVE_RADAR_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_RADAR_DIR / "reports").mkdir(exist_ok=True)
(DRIVE_RADAR_DIR / "backups").mkdir(exist_ok=True)

print(f"[+] Google Drive persistent folder ready at: {DRIVE_RADAR_DIR}")

### Step 3: Install Dependencies
Installs all packages from `requirements.txt` (tested and compatible across Python 3.10–3.12 in Colab).

In [ ]:
!pip install -q -r requirements.txt

### Step 4: Run Game Radar Pipeline (Crash-Safe Local SQLite + Drive Sync)
Executes full candidate enrichment (`--all-candidates`). Periodic checkpoints and a guaranteed `finally` block sync `radar.db` and HTML/TXT/CSV reports back to Google Drive.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Paths
LOCAL_DB = Path("/content/radar.db")
DRIVE_DB = DRIVE_RADAR_DIR / "radar.db"
REPORTS_DIR = Path("reports")
DRIVE_REPORTS_DIR = DRIVE_RADAR_DIR / "reports"

# Copy existing DB from Drive to local fast storage if available
if DRIVE_DB.exists() and DRIVE_DB.stat().st_size > 0:
    print(f"[*] Found existing database on Drive ({DRIVE_DB.stat().st_size / (1024 * 1024):.2f} MB).")
    print(f"[*] Copying {DRIVE_DB} -> {LOCAL_DB} for fast local execution...")
    shutil.copy2(DRIVE_DB, LOCAL_DB)
else:
    print("[*] No prior database found on Drive. Starting with fresh local SQLite database.")

def sync_to_drive():
    """Flush local database and generated reports to persistent Google Drive."""
    try:
        if LOCAL_DB.exists() and LOCAL_DB.stat().st_size > 0:
            # Use atomic copy via temporary file to prevent partial writes
            tmp_file = DRIVE_RADAR_DIR / "radar.db.tmp"
            shutil.copy2(LOCAL_DB, tmp_file)
            tmp_file.replace(DRIVE_DB)
            print(f"[+] Synced radar.db ({DRIVE_DB.stat().st_size / (1024 * 1024):.2f} MB) to Google Drive.")
        
        if REPORTS_DIR.exists():
            for item in REPORTS_DIR.iterdir():
                if item.is_file():
                    shutil.copy2(item, DRIVE_REPORTS_DIR / item.name)
            print(f"[+] Synced reports to {DRIVE_REPORTS_DIR}.")
    except Exception as sync_err:
        print(f"[!] Warning: Drive sync failed: {sync_err}")

# Configure environment for fast local database
os.environ["RADAR_DB_PATH"] = str(LOCAL_DB)
os.environ["PYTHONUTF8"] = "1"

cmd = [
    sys.executable, "-m", "radar", "run",
    "--mode", "full",
    "--all-candidates",
    "--db-path", str(LOCAL_DB)
]

print(f"[*] Launching: {' '.join(cmd)}")
start_time = time.time()

try:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    last_sync = time.time()
    for line in iter(proc.stdout.readline, ''):
        print(line, end="")
        # Checkpoint sync every 10 minutes
        if time.time() - last_sync > 600:
            sync_to_drive()
            last_sync = time.time()
            
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        print(f"[!] Pipeline finished with exit code {rc}")
    else:
        print("[+] Pipeline completed successfully!")
except KeyboardInterrupt:
    print("\n[!] Interrupted by user. Performing safety sync before exit...")
except Exception as e:
    print(f"\n[!] Execution error: {e}")
finally:
    sync_to_drive()
    dur = time.time() - start_time
    print(f"[*] Total run time: {int(dur // 60)}m {int(dur % 60)}s")